# CRPWarner rerun on the main contract of each project

Runtime: **CPU, High-RAM** (Runtime > Change runtime type). A GPU is not needed.

Input: `crp_input.zip`, the output folder of `prepare_bytecode.py` renamed to `crp_input` and zipped (`mv bytecode crp_input && zip -r crp_input.zip crp_input`).

Run the cells in order. Results are saved to Google Drive after every chunk, so if the session disconnects, run all cells again and the finished chunks are skipped.

In [ ]:
# 1. Mount Drive and upload crp_input.zip (only the first time)
from google.colab import drive, files
import os, zipfile
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/crpwarner_rerun'; os.makedirs(OUT, exist_ok=True)
if not os.path.exists('/content/crp_input'):
    if os.path.exists(f'{OUT}/crp_input.zip'):
        zipfile.ZipFile(f'{OUT}/crp_input.zip').extractall('/content')
    else:
        up=files.upload()   # choose crp_input.zip
        zipfile.ZipFile('crp_input.zip').extractall('/content')
        os.system(f'cp crp_input.zip {OUT}/')
print(len(os.listdir('/content/crp_input/clean')), 'clean,', len(os.listdir('/content/crp_input/obfuscated')), 'obfuscated hex files')
print('CPU cores:', os.cpu_count())

In [ ]:
%%bash
# 2. Install Souffle 2.3 and the Gigahorse dependencies (about 3 minutes)
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y libboost-all-dev libz3-dev mcpp libncurses-dev libsqlite3-dev zlib1g-dev > /dev/null
cd /tmp
wget -q http://archive.ubuntu.com/ubuntu/pool/main/libf/libffi/libffi7_3.3-4_amd64.deb
dpkg -i libffi7_3.3-4_amd64.deb > /dev/null
wget -q https://github.com/souffle-lang/souffle/releases/download/2.3/x86_64-ubuntu-2004-souffle-2.3-Linux.deb
dpkg --force-depends -i x86_64-ubuntu-2004-souffle-2.3-Linux.deb > /dev/null 2>&1 || true
souffle --version | head -2

In [ ]:
%%bash
# 3. Get CRPWarner and build the Souffle functors
set -e
cd /content
[ -d RugPull ] || git clone -q --depth 1 https://github.com/CRPWarner/RugPull.git
cd RugPull/gigahorse
rm -rf souffle-addon && git clone -q https://github.com/plast-lab/souffle-addon.git
cd souffle-addon && make > /dev/null 2>&1 && ls libfunctors.so

In [ ]:
%%bash
# 4. Warm-up: compiles the Datalog programs once (5 to 15 minutes), checked on a CRPWarner example
cd /content/RugPull/gigahorse
python3 ./gigahorse.py -C ./clients/crpwarner.dl ../dataset/groundtruth/hex/0x0414D8C87b271266a5864329fb4932bBE19c0c49.hex 2>/dev/null | tail -5

In [ ]:
# 5. Run CRPWarner on all contracts, in chunks saved to Drive
import os, glob, json, shutil, subprocess, time
J=os.cpu_count(); CH=100
for cond in ('clean','obfuscated'):
    ids=sorted(int(f[:-4]) for f in os.listdir(f'/content/crp_input/{cond}'))
    for i in range(0,len(ids),CH):
        part=ids[i:i+CH]; tag=f'{cond}_{i:04d}'; res=f'{OUT}/{tag}.json'
        if os.path.exists(res): continue
        d=f'/content/in_{tag}'; os.makedirs(d,exist_ok=True)
        for g in part: shutil.copy(f'/content/crp_input/{cond}/{g}.hex', d)
        t=time.time()
        with open(f'{OUT}/{tag}.log','w') as lg:
            subprocess.run(['python3','./gigahorse.py','-C','./clients/crpwarner.dl','-j',str(J),'-T','300','-r','/content/tmp_res.json','-w',f'/content/work_{tag}']+sorted(glob.glob(f'{d}/*.hex')),
                           cwd='/content/RugPull/gigahorse',stdout=lg,stderr=subprocess.STDOUT)
        shutil.copy('/content/tmp_res.json',res); shutil.rmtree(f'/content/work_{tag}',ignore_errors=True)
        print(tag, len(part),'contracts', f'{(time.time()-t)/60:.1f} min', flush=True)

In [ ]:
# 6. Collect the results into one CSV per condition and download them
import pandas as pd, re
for cond in ('clean','obfuscated'):
    main=pd.read_csv(f'/content/crp_input/log_{cond}.csv').set_index('GroupID')
    flags={}; timeouts=set()
    for f in sorted(glob.glob(f'{OUT}/{cond}_*.json')):
        for name,rels,*_ in json.load(open(f)):
            g=int(name.replace('.hex','')); s=set(rels)
            flags[g]=dict(HiddenMint=int('HiddenMintFunction' in s),LeakingToken=int('LeakingToken' in s),LimitingSell=int('LimitingSellOrder' in s))
        for line in open(f.replace('.json','.log')):
            m=re.match(r'\s*\d+: (\d+)\.hex completed.*Timeouts in',line)
            if m: timeouts.add(int(m.group(1)))
    rows=[]
    for g in range(1,1001):
        r=dict(GroupID=g, Artifact=(main.loc[g,'main'] if g in main.index and isinstance(main.loc[g,'main'],str) else 'no hex file'))
        if g in flags: r.update(flags[g]); r['RugPull']=bool(sum(flags[g].values())); r['Timeout']=g in timeouts
        else: r.update(HiddenMint=None,LeakingToken=None,LimitingSell=None,RugPull=False,Timeout=None)
        rows.append(r)
    df=pd.DataFrame(rows); fn=f'crpwarner_results_{cond}_maincontract.csv'; df.to_csv(fn,index=False); df.to_csv(f'{OUT}/{fn}',index=False)
    print(cond, 'analysed', len(flags), '| flagged', int(df.RugPull.sum()), '| timeouts', len(timeouts))
    files.download(fn)